# Document Loading

In [1]:
import hashlib # Used for creating chunk id
import numpy as np # Used for images .
import pymupdf # For PDF loading
from pathlib import Path # For Loading the directory or PDF path .
from typing import List,Dict
import traceback # Used for error handling .

In [2]:
# The following function is used to check if the image has detailing since if there is no much variance(Basically flat image) it removes it .
# threshold -> minimum variance value .
def is_low_variance(pix, threshold: int = 5) -> bool:

    try:
        if pix is None or pix.samples is None: # Checking if image is empty .
            return True

        samples = np.frombuffer(pix.samples, dtype=np.uint8) # Storing pixels (raw bytes) into numpy array .

        if len(samples)==0: # Checking if image array is empty .
             return True

        if pix.n >= 3:
            samples = samples.reshape(-1, pix.n)[:, :3].mean(axis=1) # Converting to grayscale

        return samples.std() < threshold # Returns True if standard variance is less than threshold .
    except Exception as e:
        print(f"Error in is_low_variance : {e}")
        return True

In [3]:
# The following function checks if image is mostly white .
# threshold -> 245 -> white
# ratio -> The threshold color present in the whole picture .
def is_mostly_white(pix, threshold=245, ratio=0.98) -> bool:

    try:
        if pix is None or pix.samples is None: # Checking if image is empty .
            return True

        samples = np.frombuffer(pix.samples, dtype=np.uint8) # Storing pixels (raw bytes) into numpy array .

        if len(samples)==0: # Checking if image array is empty .
             return True

        if pix.n >= 3:
            samples = samples.reshape(-1, pix.n)[:, :3].mean(axis=1)
        white_pixels = np.sum(samples > threshold) # Number of nearby white color variants in image .

        return (white_pixels / len(samples)) > ratio # Returns false if most of the pixels are white (Less than threshold) .
    except Exception as e:
        print(f"Error in is_mostly_white : {e}")
        return True

In [4]:
# Function to load all the PDF's from Directory .
def loading_pdf(dir_path:str='../data/pdf')->List[Dict]: # Return type .

    dir_path=Path(dir_path) # Loading directory Path

    if not dir_path.is_dir(): # Checking if the directory is valid .
        raise NotADirectoryError(f"{dir_path} is a invalid directory .")

    print(f"The directory path is : {dir_path} .")
    pdf_files=list(dir_path.rglob("*.pdf")) # Storing all the PDF's path into list .
    print(f"Number of PDF's in directory is {len(pdf_files)}")

    if len(pdf_files)==0: # Checking if any PDFs exists in the directory .
        print('No documents in the dir_path')
        return []

    # All these variables used for stats check at the end .
    all_pdf_size=0.0
    all_pages=[]
    failed_pdf=[]

    print("="*20,"PDF LOAD SUMMARY","="*20)
    print("-"*45)

    for serial,pdf_path in enumerate(pdf_files,start=1): # Iterating through all the PDFs in directory .
        print(f"{serial} ---> Loading {pdf_path.name} ")
        pdf_size_bytes=pdf_path.stat().st_size
        pdf_size_mb=pdf_size_bytes/(1024**2) # Calculating size of the PDF .
        print(f"File size : {pdf_size_mb:.3f} MB")

        pdf=None # For cleanup
        try:
            image_dir= Path('../data/images_pymupdf') / pdf_path.stem # Directory for storing images in the PDF .
            image_dir.mkdir(parents=True,exist_ok=True)

            pdf=pymupdf.open(filename=pdf_path,filetype="pdf") # Loading PDF .

            for page_num,page in enumerate(pdf,start=1):
                text_blocks=[] # Used for storing details about block of a page .
                page_images=[] # Used for storing images of current page .
                seen_xrefs=set()

                images=page.get_images(full=True) # Extracting images .

                for img_index,img in enumerate(images): # Extracting images .
                    pix=None # For cleanup

                    try:
                        if img[1]!=0: # Skip soft mask . soft mask -> Transparency layer
                            continue

                        xref=img[0]

                        if xref in seen_xrefs: # Checking if the same images are being stored
                            continue

                        seen_xrefs.add(xref)
                        rects=page.get_image_rects(xref) # Used for getting image edges .

                        if not rects: # Checking if coordinates or image is empty .
                            continue

                        pix=pymupdf.Pixmap(pdf,xref) # xref is used to find position if image in PDF .

                        if pix.width<50 or pix.height<50: # Removing very tiny images .
                            pix=None
                            continue

                        if pix.alpha and pix.samples is not None: # Removing fully transparent images .
                            if max(pix.samples)==0 and len(pix.samples)>0:
                                continue

                        if pix.n > 4:
                            pix = pymupdf.Pixmap(pymupdf.csRGB, pix)

                        if is_mostly_white(pix): # Checking if the image is mostly white .
                            pix=None
                            continue

                        if is_low_variance(pix): # Checking if its a flat image .
                            pix=None
                            continue

                        img_path=image_dir/f"page_{page_num}_img_{img_index}.png" # Location for storing images in local disk .
                        pix.save(img_path) # Saving images in local disk .
                        pix=None

                        rect=rects[0] # Changed the method since we needed only approx coordinates and not all approx coordinates to be merged , using a FOR loop made multiple copy of image .
                        page_images.append({
                            "image_id":f"{pdf_path.stem}_p{page_num}_i{img_index}",
                            "path":str(img_path),
                            "page":page_num,
                            "bbox":[rect.x0,rect.y0,rect.x1,rect.y1]
                        }) # For metadata .

                    except Exception as img_error:
                        print(f"Error processing image {img_index}")

                    finally:
                        if pix is not None:
                            pix=None

                # Following loop is to extract texts from a page .
                blocks=sorted(page.get_text("blocks"),key=lambda b:(b[1],b[0]))
                for block_id,b in enumerate(blocks):
                    x0,y0,x1,y1,text=b[:5] # Coordinates and text of text block .
                    text=text.strip()
                    if len(text) < 20: # If texts are smaller it is removed since smaller texts can not be very useful .
                        continue

                    block_bbox=[x0,y0,x1,y1] # Coordinates of the textblocks . Used while checking relevance of image and text .

                    text_blocks.append({
                        "block_id":block_id,
                        "text":text,
                        "bbox":block_bbox,
                        "page":page_num,
                    }) # Used while appending metadata .

                all_pages.append({
                    "source":pdf_path.name,
                    "page":page_num,
                    "text_blocks":text_blocks,
                    "images":page_images
                })
            all_pdf_size+=pdf_size_mb
            pdf.close()
            print("-"*45)
        except Exception as e:
            print(f" Error loading {pdf_path.name} . Error {e}") # Exception handling .
            failed_pdf.append(pdf_path.name) # Storing the PDF failed to load .
            traceback.print_exc() # Used to trace failures similar to python interpreter stack trace .
        finally:
            if pdf is not None and not pdf.is_closed:
                pdf.close()

    # Some stats of Loading all the PDF in a directory .
    print(f"Total size of all the PDF's are : {all_pdf_size:.3f} MB")
    print(f"Total Number of pages extracted : {len(all_pages)} ")
    print("-"*45)

    # Printing all the PDF which where not able to load .
    if failed_pdf:
        print(f"Failed PDF : {failed_pdf}")

    return all_pages # Returning the loaded pages .

In [5]:
pages=loading_pdf(dir_path="../data/pdf")

The directory path is : ..\data\pdf .
Number of PDF's in directory is 3
==================== PDF LOAD SUMMARY ====================
---------------------------------------------
1 ---> Loading highlights_of_hubbles_exploration_of_the_universe.pdf 
File size : 9.853 MB
---------------------------------------------
2 ---> Loading mars-science-laboratory.pdf 
File size : 1.442 MB
---------------------------------------------
3 ---> Loading Voyager Grand Tour PDF.pdf 
File size : 0.666 MB
---------------------------------------------
Total size of all the PDF's are : 11.961 MB
Total Number of pages extracted : 30 
---------------------------------------------


# Chunking

In [6]:
from langchain_core.documents import Document # Datatype of a block or a chunk .
from typing import List # Used to store list of Documents or to specify return type .
from typing import Tuple

In [7]:
# The following function is to calculate distance between two blocks and a threshold is set such that if two blocks are far those both blocks are separated with different chunks .
def vertical_gap(block1,block2)->float:
    return block2["bbox"][1]-block1["bbox"][3] # Distance between bottom of block 1 and top of block 2.

In [8]:
# The following function is used for getting outermost edge of all the chunks combined .
def merge_bbox(blocks):
    if not blocks:
        return None

    return(
        min(b["bbox"][0] for b in blocks), # x0 left
        min(b["bbox"][1] for b in blocks), # y0 top
        max(b["bbox"][2] for b in blocks), # x1 right
        max(b["bbox"][3] for b in blocks)  # y1 bottom
    )

In [9]:
# The following function creates a constant chunk id for same text .
def stable_chunk_id(source: str, page_num:int,text:str)->str:
    h=hashlib.md5(text.encode("utf-8")).hexdigest()[:8]
    return f"{source}_p{page_num}_c{h}"

In [10]:
# The following function is used to check if a block is relevant to another block using coordinates .
def bbox_overlap(a,b)->bool:
    return not(
        a[2] < b[0] or # right edge of a and left edge of b (here we are considering a is at left and b is right side of a .)
        a[0] > b[2] or # left edge of a and right edge of b (here we are considering b is at left and a is right side of b .)
        a[3] < b[1] or # bottom edge of a and top edge of b (here we are considering a is at top and b is below of a .)
        a[1] > b[3]    # top edge of a and bottom edge of b (here we are considering b is at top and a is below of b .)
    )

In [11]:
# The following function helps to identify if the text block near the image is caption of the image based on the coordinates and length of the text .
def is_caption_block(text_block:Dict,image:Dict,max_words:int=60,max_vertical_dist:int=80)->bool:

    text=text_block.get("text","") # Text

    if not text or len(text.split()) > max_words: # Checking if the text is large .
        return False

    tb_bbox=text_block.get("bbox") # Fetching text block bbox .
    im_bbox=image.get("bbox") # Fetching image block bbox .

    if not tb_bbox or not im_bbox: # Checking if image bbox is empty .
        return False

    tb_x0,tb_y0,tb_x1,tb_y1=tb_bbox # Coordinates of text block .
    im_x0,im_y0,im_x1,im_y1=im_bbox # Coordinates of image .

    horizontal_overlap=not(tb_x1 < im_x0 or tb_x0 > im_x1) # Check for horizontal overlap .

    vertical_distance=min( # Vertical distance between text block and image .
        abs(tb_y0 - im_y1),
        abs(im_y0 - tb_y1)
    )
    return horizontal_overlap and vertical_distance <= max_vertical_dist # Returns false if any condition fails .

In [12]:
# The following function get images that overlap with a chunk's bbox .
def get_overlapping_images(chunk_bbox:Tuple,page_images:List[Dict],vertical_tolerance:int =200,horizontal_tolerance:int=50) ->List[str]:
    if not chunk_bbox or not page_images:
        return []

    overlapping_ids=[]

    chunk_x0, chunk_y0, chunk_x1, chunk_y1 = chunk_bbox

    for img in page_images:
        img_bbox=img.get("bbox")
        if not img_bbox :
            continue

        img_x0, img_y0, img_x1, img_y1 = img_bbox

        # Check for horizontal overlap or proximity
        horizontal_overlap = not (
            chunk_x1 + horizontal_tolerance < img_x0 or
            chunk_x0 > img_x1 + horizontal_tolerance
        )

         # Calculate vertical distance between chunk and image
        vertical_distance = min(
            abs(chunk_y0 - img_y1),  # Distance from chunk top to image bottom
            abs(img_y0 - chunk_y1),  # Distance from image top to chunk bottom
            abs(chunk_y0 - img_y0),  # Distance between tops
            abs(chunk_y1 - img_y1)   # Distance between bottoms
        )

        # Method 1: Check if horizontally aligned and vertically close
        if horizontal_overlap and vertical_distance <= vertical_tolerance:
            overlapping_ids.append(img["image_id"])
        # Method 2: Or if they actually overlap perfectly
        elif bbox_overlap(chunk_bbox, img_bbox):
            overlapping_ids.append(img["image_id"])

    return overlapping_ids

In [13]:
# Build image objects , mainly used for embedding using openclip and also can be used parallely with text block in graph db .
def build_image_objects(pages:List[Dict])->List[Dict]:

    image_objects=[]

    # Getting text block and images from the page .(For reference this are the pages already loaded from pdf_loading .)
    for page in pages:
        source=page.get("source","unknown")
        page_num=page.get("page",0)
        text_blocks=page.get("text_blocks",[])
        images=page.get("images",[])

        # For every text block in page checking if the image is relevant or caption to it .
        for img in images:
            caption_blocks=[
                tb["text"] for tb in text_blocks
                if tb.get("text") and is_caption_block(tb,img)
            ]

            caption_text=" ".join(caption_blocks).strip() or None # Caption gathered from text blocks .

            # Appending image objects .
            image_objects.append({
                "image_id":img.get("image_id","unknown"),
                "type":"image",
                "modality":"vision",
                "source":source,
                "page_num":page_num,
                "bbox":img.get("bbox"),
                "path":img.get("path"),
                "caption_text":caption_text
            })

    return image_objects # List of image objects with relevant metadata .

In [14]:
# The following function is used create a chunk by adding
def build_chunk(source:str,page_num:int,blocks:List,page_images:List[Dict],related_image_ids:List[str] | None=None)->Document:

    if not blocks:
        return None

    chunk_text="\n".join(b.get("text","") for b in blocks) # Combing all the texts from the blocks .

    if not chunk_text.strip(): # Checking if the text blocks are empty .
        return None

    chunk_bbox=merge_bbox(blocks) # Used to get overall chunk coordinates .

    if related_image_ids is None: # If no related images then fetch related images based on bbox overlap .
        related_image_ids=get_overlapping_images(chunk_bbox,page_images)

    return Document(
        page_content=chunk_text,
        metadata={
            "source":source,
            "page_num":page_num,
            "bbox":chunk_bbox,
            "chunk_id":stable_chunk_id(source,page_num,chunk_text),
            "related_image_ids":related_image_ids or []
        }
    ) # Adding metadata .

In [15]:
# The following function is main chunking strategy ,it uses bbox , max characters to chunk different blocks together .
# max_chars -> maximum characters in a single chunk .
# (I guess we can replace with token based chunking using tiktoken need to check on that .)

# max_vertical_gap -> maximum vertical height between two blocks .
# Calculated using bbox .

# overlap_chars -> number of characters to overlap between consecutive chunks
# (0 means no overlap – same behavior as original bbox_chunker)

def bbox_chunker(
    pages: List[Dict],
    max_chars: int = 1000,
    max_vertical_gap: int = 40,
    overlap_chars: int = 0
) -> List[Document]:

    all_chunks = []  # Used to store chunks .

    # Extracting data from a dictionary .
    for page in pages:
        source = page.get("source", "")
        page_num = page.get("page", 0)
        blocks = page.get("text_blocks", [])
        page_images = page.get("images", [])  # Getting images in a page .

        if not blocks:
            continue

        i = 0
        while i < len(blocks):

            current_blocks = []  # Used to store blocks to store in a chunk .
            current_len = 0      # Calculating maximum characters in chunk .
            start_i = i          # Used to prevent infinite loop during overlap

            # Building a chunk until max_chars or vertical gap threshold
            while i < len(blocks):
                block = blocks[i]
                text = block.get("text", "")

                if not text:
                    i += 1
                    continue

                block_len = len(text)  # Calculating characters in a single block .

                if current_blocks:
                    gap = vertical_gap(current_blocks[-1], block)
                else:
                    gap = 0  # Basically first block of a page .

                # Checking conditions for creating a chunk .
                # (Basically threshold that chunk should have certain number of characters and text block distances .)
                if current_blocks and (
                    current_len + block_len > max_chars
                    or gap > max_vertical_gap
                ):
                    break

                current_blocks.append(block)
                current_len += block_len
                i += 1

            # Creating chunk from collected blocks .
            if current_blocks:
                chunk = build_chunk(source, page_num, current_blocks, page_images)
                if chunk:
                    all_chunks.append(chunk)  # Creating a chunk and appending it .

            # Step back to create overlap for next chunk .
            if overlap_chars > 0 and i < len(blocks):
                overlap_len = 0
                step_back = 0

                # Counting how many blocks to include in overlap .
                for j in range(len(current_blocks) - 1, -1, -1):
                    block_text = current_blocks[j].get("text", "")
                    if overlap_len + len(block_text) <= overlap_chars:
                        overlap_len += len(block_text)
                        step_back += 1
                    else:
                        break

                # Move index back safely (avoid infinite loop) .
                i = max(start_i + 1, i - step_back)

    print(
        f"{len(all_chunks)} chunks were created from {len(pages)} pages "
        f"(max_chars={max_chars}, overlap_chars={overlap_chars})"
    )

    return all_chunks

In [16]:
image_objects=build_image_objects(pages)

In [17]:
chunks=bbox_chunker(pages)

97 chunks were created from 30 pages (max_chars=1000, overlap_chars=0)


# Embedding

In [18]:
import numpy as np # Used for storing embeddings .
import torch # For device selection , model execution and tensor operation .
from PIL import Image # Used for image creation and other image operation .
import open_clip # Image embedding model .
from typing import List, Dict # Used for type return .

In [19]:
# Used for loading embedding model and embedding images with their caption .
class ImageEmbeddingModel:
    def __init__(self,model_name:str="ViT-B-32",pretrained:str="laion2b_s34b_b79k"):

        self.device="cuda" if torch.cuda.is_available() else "cpu" # Used to check if the machine has GPU if not assign CPU as device for working .
        try:
            self.model,self.preprocess,_=open_clip.create_model_and_transforms(
                model_name=model_name,
                pretrained=pretrained
            ) # Getting essential function from model .
            self.tokenizer=open_clip.get_tokenizer(model_name)

        except Exception as e:
            raise RuntimeError(f"Failed to load OpenClip model : {e}")


        self.model=self.model.to(self.device) # Device selection for model operations .
        self.model.eval() # Loading model

        # Displaying device on which model will run .
        if self.device=="cuda":
            print(f"OpenClip running on {torch.cuda.get_device_name(0)} .")
        else:
            print("OpenClip running on CPU .")
        print(f"Embedding dimension of {model_name} is {open_clip.get_model_config(model_name).get('embed_dim')}")

    # Using no grad since by default torch assumes it is used for training model and do various backpropagation tasks which are not required here and only forward pass is required .
    @torch.no_grad()
    def embed_image(self,image_objects:List[Dict])->np.ndarray:
        if not image_objects:
            raise ValueError("No images in the image object .")

        image_objects_embeddings=[] # Used for storing image embeddings .

        for image_object in image_objects: # Iterating through image objects .
            image_path=image_object.get("path") # Fetching location of image.

            try:
                image=Image.open(image_path).convert("RGB") # Converting RGB if image is not already in RGB .
            except Exception as e:
                print(f"Failed to load {image_path} : {e}")
                continue

            image_tensor=self.preprocess(image).unsqueeze(0).to(self.device) # Converting into tensor since openclip cant embedd other inputs .

            emb_imag=self.model.encode_image(image_tensor) # Embedding images
            emb_imag=emb_imag/emb_imag.norm(dim=-1,keepdim=True) # Normalizing images

            emb_imag=emb_imag.cpu().numpy()[0] # Converting tensor output to numpy array . Using .cpu since numpy cannot access gpu memory ,it should be in ram memory then numpy can access memory and can convert it into numpy array .

            caption = image_object.get("caption_text") or ""
            caption = caption.strip() if isinstance(caption, str) else ""
            if caption: # If caption is available for the image then embedd it .

                tokens=self.tokenizer([caption]).to(self.device) # Used to convert into tensor .

                emb_text=self.model.encode_text(tokens) # Embedding text or tensor value .
                emb_text=emb_text/emb_text.norm(dim=-1,keepdim=True) # Normalizing values

                emb_text=emb_text.cpu().numpy()[0] # Converting tensor to numpy array .

                fused=0.7*emb_imag+0.3*emb_text # Fusing image and caption together .It does not change the meaning . Adding more weight to image compared to caption .

                norm= np.linalg.norm(fused)
                if norm > 0:
                    fused=fused / norm # Normalizing fused values .

                image_objects_embeddings.append(fused) # Appending embeddings .

            else:
                image_objects_embeddings.append(emb_imag) # If no captions only images are embedded and appended .

        if not image_objects_embeddings: # Checking if image embeddings is empty .
            raise ValueError("No valid images were processed. All images failed to load .")

        return np.vstack(image_objects_embeddings) # Returning all the embeddings .

    @torch.no_grad()
    def embed_query(self,query:str)->np.ndarray:
        if not isinstance(query,str): # Checking if query is valid .
            raise TypeError("Query must be a string .")

        if not query.strip():
            raise ValueError("Please give a valid prompt .")

        tokens=self.tokenizer([query]).to(self.device) # Used to convert into tensor .
        query_embedding=self.model.encode_text(tokens) # Embedding text or tensor value .
        query_embedding=query_embedding/query_embedding.norm(dim=-1,keepdim=True) # Normalizing values
        query_embedding=query_embedding.cpu().numpy()[0] # Converting tensor to numpy array .

        return query_embedding # Returning all the embeddings .

In [20]:
from sentence_transformers import SentenceTransformer # Used for loading model .
import numpy as np # Used to store embedding .
import torch # Used for device selection and model execution .
from typing import List # Used for return type .
from langchain_core.documents import Document # Used for storing documents .

In [21]:
# Used for loading model and embedding text .
class TextEmbeddingModel:

    def __init__(self,model_name:str= "BAAI/bge-base-en-v1.5"):
        self.device="cuda" if torch.cuda.is_available() else "cpu" # Device check .

        try:
            self.model=SentenceTransformer(model_name_or_path=model_name,device=self.device) # Loading model .
        except Exception as e:
            raise RuntimeError(f"Failed to load SentenceTransformer model: {e}")

        self.query_prefix="Represent this sentence for searching relevant passages: "

        if self.device=="cuda":
            print(f"BGE running on {torch.cuda.get_device_name(0)} .")
        else:
            print("BGE running on CPU .")
        print(f"Embedding dimension of {model_name} is {self.model.get_sentence_embedding_dimension()}")

    @torch.no_grad()
    def embed_documents(self,documents:List[Document])->np.ndarray:
        if not documents:
            raise ValueError("No documents to embed .")

        texts=[]
        for doc in documents: # Checking if data is available in documents .
            if hasattr(doc,"page_content") and doc.page_content:
                content=doc.page_content.strip()
                if content:
                    texts.append(content)

        if not texts:
            raise ValueError("No valid document content to embed .All documents are empty .")

        # Embedding texts .
        # sentences -> input /texts
        # batch_size -> Number of inputs embedding a single time .
        # convert_to_numpy -> Convert output to numpy array .
        # normalize_embeddings -> Normalizing all the out vectors .
        text_embeddings=self.model.encode(
            sentences=texts,
            batch_size=32,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        return text_embeddings # Return embeddings .

    # The following function is used to create embedding for user input prompt .
    @torch.no_grad()
    def embed_query(self,query:str)->np.ndarray:
        if not isinstance(query, str): # Checking if query is valid .
            raise TypeError("Query must be a string.")

        if not query.strip(): # Checking if the prompt is not empty .
            raise ValueError("Given prompt is not valid .")

        query=self.query_prefix + query # Adding a prefix to prompt since BGE is instruction based embedding model .(Only for prompt not mandatory for documents)

        query_embedding=self.model.encode( # Embedding query .
            sentences=query,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )

        return query_embedding # Returning embedded values .

In [22]:
image_model = ImageEmbeddingModel() # Loading model .

OpenClip running on CPU .
Embedding dimension of ViT-B-32 is 512


In [23]:
text_model = TextEmbeddingModel() # Loading model .

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BGE running on CPU .
Embedding dimension of BAAI/bge-base-en-v1.5 is 768


In [24]:
image_embeddings=image_model.embed_image(image_objects=image_objects)

In [25]:
text_embeddings=text_model.embed_documents(chunks)

# VectorStore

In [26]:
import json # Used for receiving image object or document while creating a hash id .
from typing import List,Dict,Union,Optional # For datatype of a variable .
import chromadb # Used for creating a vectorDB .
import os # For getting directory path for storing vector database .
import hashlib # Used for creating hashid for documents
import numpy as np
from langchain_core.documents import Document

In [27]:
# The following function is used to create hashid using content of a document or image object .
def stable_hash(obj:dict|str)->str:
    if isinstance(obj,dict):
        obj=json.dumps(obj,sort_keys=True,ensure_ascii=False)
    elif not isinstance(obj,str):
        raise TypeError(f"stable_hash expects dict or str, got {type(obj)}")

    return hashlib.sha256(obj.encode("utf-8")).hexdigest()

In [28]:
# The following function is used to remove None type and replace it with empty string since chromadb cannot store type None .
def sanitize_metadata(metadata: dict) -> dict:
    if not isinstance(metadata,dict): # Type validation .
        raise TypeError(f"sanitize_metadata expects dict , got {type(metadata)}")

    clean = {}
    for k, v in metadata.items():
        if v is None:
            clean[k] = "" # None -> Empty string
        elif isinstance(v, (str, int, float, bool)):
            clean[k] = v # Keep it as it is .
        else:
            clean[k] = str(v) # Convert unknown type to string .
    return clean # Return sanitized metadata .


In [29]:
# Used for initializing vectorDB and also store data in collection .
class VectorStore:
    def __init__(self,collection_name:str,directory:str="../data/database"):
        if not collection_name or not isinstance(collection_name,str):
            raise ValueError("Collection name must be a non-empty string .")

        self.collection_name=collection_name # Collection name .
        self.persistent_directory=directory # Directory to store database .
        self.collection=None # Collection , used to store data .
        self.client=None # Used to connect database .
        self.initialize_store() # Initializing vectordb .

    def initialize_store(self):
            try:
                os.makedirs(name=self.persistent_directory,exist_ok=True) # Checking if directory exists ,if not creating one .
                self.client=chromadb.PersistentClient(path=self.persistent_directory)

                if self.collection_exists(self.collection_name): # Checking if the collection exists , if collection exists then loading it .
                    print(f"Loading collection {self.collection_name} from database .")
                    self.collection=self.client.get_collection(self.collection_name)

                else: # If collection does not exist then creating it .
                    print(f"New collection {self.collection_name} created in database .")
                    self.collection=self.client.create_collection(name=self.collection_name,metadata={"hnsw:space":"cosine"})

                print(f"Vector store initialized .") # Success message is vector store initialized .
                print(f"Existing documents in collection : {self.collection.count()}")

            except Exception as e: # Exception handling .
                raise RuntimeError(f"Could not initialize vector store : {e}") from e

    # The following function is used to add data and its embedding to a collection .
    def add_documents(self,documents:List[Union[Dict,Document]],embeddings:np.ndarray):
        if not self.collection: # Checking if collection is initialized .
            raise RuntimeError("Collection is not initialized .")

        if not documents:
            raise ValueError("Documents list is empty .")

        if len(documents)!=len(embeddings): # Checking if number of documents and embeddings are same .
            raise ValueError(f"Number of documents ({len(documents)}) does not match embeddings ({len(embeddings)}).") # If not raise an error

        ids,metadatas,texts=[],[],[] # Used to store main content and metadata .

        for doc in documents:
            if isinstance(doc,Document): # For text embeddings . Document type .
                content=doc.page_content.strip()
                if not content:
                    continue

                metadata=doc.metadata or {}
                metadata=sanitize_metadata(metadata)

                hash_input={ # Data used for creating hashid .
                    "content":content,
                    "source":metadata.get("source"),
                    "page":metadata.get("page_num","")
                }
                doc_id=stable_hash(hash_input)

                texts.append(content) # Appending data to store it in vectorDB
                metadatas.append(metadata)
                ids.append(doc_id)

            elif isinstance(doc,dict): # For image embeddings . Dict type .
                bbox = doc.get("bbox")
                image_metadata = {
                "image_path": doc.get("path", ""),
                "caption_text": doc.get("caption_text", ""),
                "bbox": json.dumps(bbox) if bbox is not None else "",
                }

                image_metadata = sanitize_metadata(image_metadata) # Removing None or unknown datatype .

                hash_input = { # Used for hashid .
                "image_path": image_metadata["image_path"],
                "bbox": image_metadata["bbox"],
                "caption": image_metadata["caption_text"],
                }

                doc_id=stable_hash(hash_input)

                texts.append(doc.get("caption_text","")) # Appending data to store it in vectorDB
                metadatas.append(image_metadata)
                ids.append(doc_id)

            else:
                raise TypeError(f"Unsupported document type :{type(doc)}") # If the input neither Document nor Dict .

        if not ids:
            print("No valid documents to process .")

        existing_ids=set( # Used to check if the document is previously added .
            self.collection.get(include=[])["ids"]
        )

        seen=set()
        new_indices=[]

        for i, doc_id in enumerate(ids): # Check for redundant documents .
            if doc_id in existing_ids:
                continue
            if doc_id in seen:
                continue
            seen.add(doc_id)
            new_indices.append(i)

        if not new_indices: # Checking if there are new documents to add to collection .
            print("No new documents to add")
            return

        self.collection.add( # Adding new documents to collection
            ids=[ids[i] for i in new_indices],
            documents=[texts[i] for i in new_indices],
            metadatas=[metadatas[i] for i in new_indices],
            embeddings=[embeddings[i].tolist() for i in new_indices]
        )
        print(f"Added {len(new_indices)} new documents to collection .")

    # The following function is used to check if the collection exists .
    def collection_exists(self,collection_name:str)->bool:
        collection_in_db=self.client.list_collections()
        return any(col.name==collection_name for col in collection_in_db )

    # The following function is used get queries and get relevant documents
    def query(self,query_embedding:np.ndarray,k:int=5,where:Optional[Dict]=None):

        if not self.collection: # Checking is collection is initialized .
            raise RuntimeError("Collection is not initialized .")

        if query_embedding.ndim!=1: # Checking if query is 1 dimensional .
            raise ValueError("Query embedding must be a 1D vector .")

        if not isinstance(k,int) or k <= 0:
            raise ValueError(f"k must be a positive integer ,got {k} .")

        results=self.collection.query( # Storing results .
            query_embeddings=[query_embedding.tolist()],
            n_results=k, # Getting top results .
            where=where, # Acts as a filter .
            include=["documents","metadatas","distances"] # Including the following data .
        )
        return results

    # The following function is used get stats about the collection .
    def get_collection_stats(self)->Dict:
        if not self.collection:
            raise RuntimeError("Collection is not initialized .")

        return {
            "name":self.collection_name,
            "count":self.collection.count(),
            "directory":self.persistent_directory
        }

    # The following collection is used to delete current collection .
    def delete_collection(self)->None:
        if not self.collection:
            raise RuntimeError("Collection is not initialized .")

        self.client.delete_collection(name=self.collection_name)
        self.collection=None
        print(f"Collection {self.collection_name} deleted .")

In [30]:
image_vector_store=VectorStore(collection_name="OpenClip_embeddings")

New collection OpenClip_embeddings created in database .
Vector store initialized .
Existing documents in collection : 0


In [31]:
text_vector_store=VectorStore(collection_name="BGE_embeddings")

New collection BGE_embeddings created in database .
Vector store initialized .
Existing documents in collection : 0


In [32]:
text_vector_store.add_documents(documents=chunks,embeddings=text_embeddings)

Added 97 new documents to collection .


In [33]:
image_vector_store.add_documents(documents=image_objects,embeddings=image_embeddings)

Added 99 new documents to collection .


# Retrieval

In [34]:
from PIL import Image
import os
from typing import List,Dict,Optional

# Context formatter

In [35]:
# Used to format retrieved docs from retrieval output .
class ContextFormatter:

    def __init__(self,max_text_chunks:int=3,max_images:int=1,text_distance_threshold:float=0.65,image_distance_threshold:float=0.95):
        if not isinstance(max_text_chunks, int) or max_text_chunks <= 0: # Threshold value validation .
            raise ValueError(f"max_text_chunks must be a positive integer, got {max_text_chunks}.")

        if not isinstance(max_images, int) or max_images <= 0:
            raise ValueError(f"max_images must be a positive integer, got {max_images}.")

        if not isinstance(text_distance_threshold, (int, float)) or text_distance_threshold < 0:
            raise ValueError(f"text_distance_threshold must be a non-negative number, got {text_distance_threshold}.")

        if not isinstance(image_distance_threshold, (int, float)) or image_distance_threshold < 0:
            raise ValueError(f"image_distance_threshold must be a non-negative number, got {image_distance_threshold}.")

        self.max_text_chunks=max_text_chunks # Limiting number of chunks .
        self.max_images=max_images # Limiting number of max images .
        self.text_distance_threshold=float(text_distance_threshold) # Threshold for filtering chunks .
        self.image_distance_threshold=float(image_distance_threshold) # Threshold for filtering images .

    # Removing unwanted details from results .
    def _flatten_results(self,results:Dict)->List[Dict]:

        if not results or not isinstance(results,dict): # Checking if data is empty .
            return []

        documents = results.get("documents", [[]])
        metadatas = results.get("metadatas", [[]])
        distances = results.get("distances", [[]])

        # Check if we have valid nested lists .
        if not documents or not documents[0]:
            return []

        docs = documents[0] if len(documents) > 0 else []
        metas = metadatas[0] if len(metadatas) > 0 else []
        dists = distances[0] if len(distances) > 0 else []

        # Ensure all lists have the same length .
        min_length = min(len(docs), len(metas), len(dists))
        if min_length == 0:
            return []

        flattened=[]
        for i in range(min_length):
            flattened.append({ # Appending required data .
                "text":docs[i],
                "metadata":metas[i],
                "distance":float(dists[i])
            })

        return flattened # Returning flattened data .

    # The following function is used to load image from path .
    def _load_image(self,image_path:str) -> Optional[Image.Image]:

        if not image_path or not isinstance(image_path, str): # Checking is image path is valid .
            return None

        if not os.path.exists(image_path):
            print(f"Warning: Image path does not exist: {image_path}")
            return None

        try:
            return Image.open(image_path).convert("RGB") # Returning loaded image .
        except Exception as e:
            print(f"Failed to load image {image_path}: {e}")
            return None

    # The following function is filter text chunks based on distance .
    def _select_text_chunks(self,text_results:Dict)->List[Dict]:
        items=self._flatten_results(text_results) # Removing unwanted data from results .

        if not items: # Checking if items is empty .
            return []

        filtered=[
            i for i in items
            if i["distance"] <= self.text_distance_threshold # Filtering based on distance .
        ]

        if not filtered: # If there is no chunks (All the chunks distance greater than threshold) just return retrieved chunks .

            items.sort(key=lambda x:x["distance"])
            return items[: self.max_text_chunks] # Returning max number of chunks .

        filtered.sort(key=lambda x:x["distance"])

        return filtered[: self.max_text_chunks] # Returning filtered chunks .

    # The following function is to format text in a data .
    def _format_text_context(self,text_items:List[Dict])->str:
        if not text_items or not isinstance(text_items, list): # Input validation .
            return ""

        lines=[]

        for idx,item in enumerate(text_items,start=1): # Fetching required data .
            if not isinstance(item, dict):
                continue

            meta=item["metadata"] or {}
            source=meta.get("source","unknown")
            page=meta.get("page_num","N/A")

            text=item["text"].strip()

            if not text:
                continue

            lines.append(
                f"[{idx}] {text}\n"
                f"(Source: {source}, page {page})" # Attaching source and page number .
            )

        return "\n\n".join(lines) # Returning formatted text .

    # The following function is select images from retrieved output .
    def _select_images(self,image_results:Dict)->List[Dict]:
        items=self._flatten_results(image_results) # Removing unwanted data from results .

        if not items:
            return []

        items=[
            i for i in items
            if i["distance"] <= self.image_distance_threshold # Filtering based on distance .
        ]

        items.sort(key=lambda x: x["distance"])

        return items[: self.max_images] # Returning filtered output .

    # The following function is used to format image caption .
    def _format_image_context(self,image_items:List[Dict])->List[Dict]:

        if not image_items or not isinstance(image_items, list):
            return []

        formatted_images=[]

        for item in image_items: # Fetching required data .
            if not isinstance(item, dict):
                continue

            meta=item.get("metadata") or {}
            image_path=meta.get("image_path")
            caption=meta.get("caption_text", "").strip()

            image=self._load_image(image_path) # Loading image .
            if image is None: # Checking if image is not empty .
                continue

            formatted_images.append({ # Attaching image and its caption .
                "image":image,
                "caption":caption
            })

        return formatted_images # Returning formatted images .

    # The following function is to format both text and images .
    def format(self,retrieval_output:Dict)->Dict:
        if not retrieval_output or not isinstance(retrieval_output, dict):
            raise ValueError("retrieval_output must be a non-empty dictionary.")

        query=retrieval_output.get("query", "") # Fetching query .

        text_items=self._select_text_chunks(
            retrieval_output.get("text_results",{}) # Selecting text chunks .
        )

        image_items=self._select_images(
            retrieval_output.get("image_results",{}) # Selecting images .
        )

        return { # Returning filtered text and images .
            "query":query,
            "text_context":self._format_text_context(text_items),
            "images":self._format_image_context(image_items)
        }

In [36]:
from sentence_transformers import CrossEncoder
from typing import List,Optional

In [37]:
# Used to rerank the chunks with relevance .
class Reranker:
    def __init__(self):# Initializing model .
        self.model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        print("Reranker initialized")

    def rerank(
        self,query: str,items: List[Dict],top_k: int = 3) -> List[Dict]:

        if not items: # Checking if chunks are empty .
            return []

        pairs = [[query, item["text"]] for item in items]
        scores = self.model.predict(pairs) # Reranking distances .

        ranked = sorted( # Ranking based of distance .
            zip(items, scores),
            key=lambda x: x[1],
            reverse=True # Since lower the distance higher the relevance .
        )

        return [item for item, _ in ranked[:top_k]] # Returning top k chunks .

In [38]:
# Used to retrieve documents from database using input query from user .
class RetrievalRag:

    def __init__(self,open_clip_embedding:ImageEmbeddingModel,bge_embedding:TextEmbeddingModel,image_vectordb:VectorStore,text_vectordb:VectorStore,reranker: Optional[Reranker] = None,formatter: Optional[ContextFormatter] = None):

        if not isinstance(open_clip_embedding, ImageEmbeddingModel):
            raise TypeError("open_clip_embedding must be an instance of ImageEmbeddingModel")
        if not isinstance(bge_embedding, TextEmbeddingModel):
            raise TypeError("bge_embedding must be an instance of TextEmbeddingModel")
        if not isinstance(image_vectordb, VectorStore):
            raise TypeError("image_vectordb must be an instance of VectorStore")
        if not isinstance(text_vectordb, VectorStore):
            raise TypeError("text_vectordb must be an instance of VectorStore")
        if reranker is not None and not isinstance(reranker, Reranker):
            raise TypeError("reranker must be an instance of ReRanker")

        self.open_clip_embedding=open_clip_embedding # Image embedding model .
        self.bge_embedding=bge_embedding # Text embedding model .
        self.image_vectordb=image_vectordb # Image database .
        self.text_vectordb=text_vectordb # Text database .
        self.reranker = reranker # Reranker
        self.formatter = formatter or ContextFormatter() # Context Formatter used to flatten documents here .

    # Note : We are embedding query from the embedding models used to embed the data in database since the dimensions of the both query and data must be same to do a semantic search .
    # The following function is used to retrieve text from database .
    def retrieve_text(self,query:str,k:int=5)->Dict:
        if not query or not query.strip(): # Query validation .
            raise ValueError("Query must be a non-empty string.")

        if not isinstance(k, int) or k <= 0:
            raise ValueError(f"k must be a positive integer, got {k}.")

        query_embedding=self.bge_embedding.embed_query(query) # Embedding query .

        results=self.text_vectordb.query(query_embedding=query_embedding,k=k)

        return results # Returning top k results .

    # The following function is used to retrieve images and its caption from database .
    def retrieve_images(self,query:str,k:int=3)->Dict:
        if not query or not query.strip(): # Query validation .
            raise ValueError("Query must be a non-empty string.")

        if not isinstance(k, int) or k <= 0:
            raise ValueError(f"k must be a positive integer, got {k}.")

        query_embedding=self.open_clip_embedding.embed_query(query) # Embedding query .

        results=self.image_vectordb.query(
            query_embedding=query_embedding,k=k
        )

        return results

    # Used to retrieve
    def retrieve(self, query: str, text_k: int = 5, image_k: int = 3, rerank_k: int = 3)->Dict:
        if not query or not query.strip(): # Checking if query is valid .
            raise ValueError("Query must be non-empty string .")

        if not isinstance(text_k, int) or text_k <= 0: # Threshold validation .
            raise ValueError(f"text_k must be a positive integer, got {text_k}.")

        if not isinstance(image_k, int) or image_k <= 0:
            raise ValueError(f"image_k must be a positive integer, got {image_k}.")

        text_results=self.retrieve_text(query=query,k=text_k)
        image_results=self.retrieve_images(query=query,k=image_k)

        if self.reranker: # Checking if reranking is initialized .
            items = self.formatter._flatten_results(text_results)
            reranked = self.reranker.rerank(
                query=query,
                items=items,
                top_k=min(rerank_k, len(items)),
            )

            # Rebuild results in original structure .
            text_results["documents"][0] = [i["text"] for i in reranked]
            text_results["metadatas"][0] = [i["metadata"] for i in reranked]
            text_results["distances"][0] = [i["distance"] for i in reranked]

        return { # Returning retrieved documents from database .
            "query":query,
            "text_results":text_results,
            "image_results":image_results
        }

In [39]:
from typing import List, Dict
import matplotlib.pyplot as plt
from PIL import Image

In [40]:
formatter=ContextFormatter() # Initializing context formatter .

In [41]:
reranker=Reranker()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker initialized


In [42]:
retrieval=RetrievalRag(open_clip_embedding=image_model,bge_embedding=text_model,image_vectordb=image_vector_store,text_vectordb=text_vector_store,reranker=reranker,formatter=formatter) # Initializing required variables .

# Testing

In [43]:
TEST_QUESTIONS = [
    # ======================
    # SECTION 1: HUBBLE SPACE TELESCOPE (Questions 1-15)
    # ======================

    # A. TEMPORAL & VISUAL REASONING
    {
        "id": 1,
        "type": "temporal_visual_analysis",
        "question": "Analyze the three images of Jupiter's Great Red Spot from 1995, 2009, and 2014 (page 7). What visual evidence demonstrates the storm's size reduction, and what physical mechanisms does the document suggest are responsible for this change?",
        "expected_images": True
    },
    {
        "id": 2,
        "type": "temporal_visual_analysis",
        "question": "Examine the HH 34 jet observations from 1994, 1998, and 2007 (page 10). What specific visual changes indicate outward motion, and how does the document explain the episodic nature of these stellar outflows?",
        "expected_images": True
    },

    # B. IMAGE–MODEL INFERENCE
    {
        "id": 3,
        "type": "model_dependency",
        "question": "Using the Cl 0024+17 galaxy cluster images (page 4), distinguish between directly observable features and model-dependent inferences. How does Hubble's imaging enable dark matter distribution mapping?",
        "expected_images": True
    },
    {
        "id": 4,
        "type": "observation_vs_inference",
        "question": "Analyze the M84 galaxy observations (page 5). What does the spectrograph image directly show, and what chain of inference leads from these observations to the black hole mass determination?",
        "expected_images": True
    },

    # C. STRUCTURE VS INTERPRETATION
    {
        "id": 5,
        "type": "image_text_alignment",
        "question": "Compare the Hubble image and illustration of TW Hydrae (page 15). Why is the gap's incomplete clearing stronger evidence for an actively forming planet than a simple empty ring would be?",
        "expected_images": True
    },
    {
        "id": 6,
        "type": "temporal_causality",
        "question": "Trace the evolution shown in the SN 1987A image sequence from 1994-2006 (page 13). How do the brightening spots relate to the supernova's explosion timeline and pre-existing circumstellar environment?",
        "expected_images": True
    },

    # D. BEYOND-VISIBLE INFERENCE
    {
        "id": 7,
        "type": "methodological_inference",
        "question": "The Fomalhaut b images (page 3) show only a faint point of light. How does the document explain Hubble's capability to determine atmospheric composition despite this limited visual information?",
        "expected_images": True
    },
    {
        "id": 8,
        "type": "morphology_classification",
        "question": "Compare the early galaxies in the Hubble Ultra Deep Field (page 4, right panel) with nearby galaxies like M104 or NGC 1300 (page 16). What morphological differences indicate cosmic evolution, and how does distance relate to lookback time?",
        "expected_images": True
    },

    # E. PHYSICAL GEOMETRY & DYNAMICS
    {
        "id": 9,
        "type": "spatial_reasoning",
        "question": "Examine the X-shaped asteroid debris field (page 9, top). What geometric features distinguish this as collision debris rather than rotational breakup or cometary activity?",
        "expected_images": True
    },

    # F. COUNTERINTUITIVE PHYSICS
    {
        "id": 10,
        "type": "counterintuitive_physics",
        "question": "The V838 Mon sequence (page 14) shows an apparently expanding shell. Explain why this is an optical illusion, and describe the actual physical process creating the observed changes.",
        "expected_images": True
    },

    # G. ADDITIONAL INTEGRATED QUESTIONS
    {
        "id": 11,
        "type": "dark_energy_expansion_synthesis",
        "question": "Using the supernova distance measurements described on page 1, explain how observations of Type Ia supernovae led to the discovery of cosmic acceleration and the dark energy hypothesis.",
        "expected_images": True
    },
    {
        "id": 12,
        "type": "subsurface_ocean_comparative_detection",
        "question": "Compare the evidence for subsurface oceans on Europa and Ganymede (pages 6-7). How do different observational techniques reveal liquid water beneath icy surfaces?",
        "expected_images": True
    },
    {
        "id": 13,
        "type": "mission_preparation_dynamics",
        "question": "Using the Pluto observations (page 8), describe how Hubble's discoveries influenced the New Horizons mission planning and what the chaotic rotation of Nix and Hydra reveals about the system's dynamics.",
        "expected_images": True
    },
    {
        "id": 14,
        "type": "star_formation_feedback_cycles",
        "question": "Analyze the 'Mystic Mountain' image (pages 10-11). How do the visible structures relate to the star formation process, and what role do stellar winds play in sculpting the nebula?",
        "expected_images": True
    },
    {
        "id": 15,
        "type": "planetary_nebula_morphological_diversity",
        "question": "Examine the planetary nebula collection (page 12). What physical mechanisms create the diverse morphologies (pinwheels, butterflies, hourglasses), and how do these shapes constrain models of stellar death?",
        "expected_images": True
    },

    # ======================
    # SECTION 2: MARS SCIENCE LABORATORY/CURIOSITY (Questions 16-19)
    # ======================

    {
        "id": 16,
        "type": "evidence_chain_synthesis",
        "question": "Using the Link outcrop image (rounded pebbles) and the John Klein drill site image (powdered interior sample), does the context enable reconstruction of a complete geological and habitability narrative — including mechanical evidence of flowing water, chemical evidence of sustained water-rock interaction, calculation of the 4.12-billion-year burial interval (4.2 Ga formation vs. 80 Ma exposure), and justification for why all four habitability factors must be considered together rather than individually?",
        "expected_images": True
    },
    {
        "id": 17,
        "type": "spatial_engineering_causal_reasoning",
        "question": "Does the Gale Crater landing analysis require extracting numerical precision data (20 km improvement vs. ±60 km prior uncertainty, ~150 km crater size), recognizing Mount Sharp's position within the crater, and constructing the causal chain that links precision landing capability to viable site selection and the ability to study environmental evolution through layered stratigraphy?",
        "expected_images": True
    },
    {
        "id": 18,
        "type": "instrumentation_epistemology",
        "question": "Does the document require distinguishing between visual imaging, chemical spectroscopy, subsurface hydrogen detection, and radiation measurement in order to explain why rock appearance alone cannot determine habitability, how complementary instruments form a hierarchical decision strategy (remote → close-up → drill → lab analysis), and why subsurface preservation and radiation effects are essential to interpreting Mars' past environments?",
        "expected_images": True
    },
    {
        "id": 19,
        "type": "atmospheric_loss_mechanism_inference",
        "question": "Does the document require synthesizing evidence from three distinct measurement contexts — (1) in-flight radiation measurements between Earth and Mars, (2) atmospheric composition analysis at the surface, and (3) the inference that Mars lost atmosphere 'by a process favoring loss from the top of the atmosphere rather than interaction with the surface' — to construct a counterintuitive physical explanation? Specifically: Why does the contrast between upper-atmosphere escape (Jeans escape, solar wind stripping) versus surface interaction (chemical weathering, mineral incorporation) matter for habitability assessment, how does current surface radiation connect to ancient atmospheric thickness, and why would radioisotope power generation data (110W declining to 100W over 2 years, plutonium-238 decay) be relevant to long-term mission planning in this context when solar panels would fail in the current thin atmosphere?",
        "expected_images": False
    },

    # ======================
    # SECTION 3: VOYAGER GRAND TOUR (Questions 20-23)
    # ======================

    {
        "id": 20,
        "type": "orbital_contingency_reasoning",
        "question": "Does the trajectory diagram and mission timeline collectively demonstrate that gravity assists redirect spacecraft paths (not merely accelerate them), that Voyager 1's close Titan flyby intentionally altered its trajectory to preclude further planetary encounters, and that Voyager 2's extended Uranus-Neptune mission was contingently enabled by a pre-calculated Saturn flyby geometry rather than post-launch reprogramming?",
        "expected_images": True
    },
    {
        "id": 21,
        "type": "temporal_spatial_operational_inference",
        "question": "Does the Family Portrait image, trajectory diagram, and power system description together require reasoning that the February 14, 1990 photograph was taken after the final planetary encounter (Neptune, August 1989) from a specific geometric vantage point, and that delaying permanent camera shutdown represented a deliberate symbolic and narrative decision rather than a purely scientific necessity?",
        "expected_images": True
    },
    {
        "id": 22,
        "type": "philosophical_probability_synthesis",
        "question": "Does the Golden Record description require reconciling the vanishingly small probability of extraterrestrial discovery (given Voyager's trajectory avoids close stellar encounters for tens of thousands of years) with the deliberate inclusion of playback instructions, analog record technology, and culturally curated content, implying that the record functions as a symbolic human self-representation as much as an interstellar communication attempt?",
        "expected_images": True
    },
    {
        "id": 23,
        "type": "boundary_detection_paradox",
        "question": "Does the Voyager Interstellar Mission description require reconciling three apparent contradictions: (1) How can Voyager 1's August 2012 boundary crossing be definitively dated when the boundary region is described as 'poorly understood' and required years of post-detection analysis? (2) Why does the document express certainty about Voyager 1's past crossing but uncertainty about Voyager 2's future crossing despite both having similar outbound trajectories?",
        "expected_images": False
    },

    # ======================
    # SECTION 4: CROSS-DOCUMENT SYNTHESIS (Questions 24-26)
    # ======================

    {
        "id": 24,
        "type": "cross_mission_habitability_epistemology",
        "question": "Comparing Hubble's Europa observations (spectroscopic water vapor detection, visual plume evidence) with Voyager's distant Europa flyby images and Curiosity's John Klein drill analysis (4.2 Ga subsurface habitability evidence), does the synthesis reveal fundamental differences in how habitability is assessed across mission types? Specifically: Why can Hubble detect current volatile activity on Europa from Earth orbit while Voyager's close flyby images alone were insufficient for habitability claims, how does the temporal preservation paradox differ between Europa's active subsurface ocean (potentially habitable now) versus Mars' ancient aqueous environment (habitable 4 billion years ago but not today), and why would Curiosity's drill-based subsurface sampling strategy be essential for Mars but unnecessary for Europa despite both requiring evidence of liquid water, chemical energy sources, and organic compounds?",
        "expected_images": True
    },
    {
        "id": 25,
        "type": "cross_mission_temporal_scale_synthesis",
        "question": "Synthesizing temporal and spatial scales across all three missions: Compare Hubble's observation of the Crab Nebula (supernova remnant from 1054 AD, ~6,500 light-years away), Curiosity's radiometric dating of Martian rocks (4.2 billion years formation age with 80 million years surface exposure), and Voyager's trajectory (40-year mission duration, will not approach another star for tens of thousands of years). How do these timescales reveal different meanings of 'ancient' and 'distant'—why does Hubble's 1054 AD observation represent recent cosmic history despite being older than any human record of Martian surface conditions, how does Curiosity's 4.2 Ga rock age compare to the 13.8 Ga age of galaxies in Hubble's Ultra Deep Field, and why is Voyager's physical 40-year journey cosmologically insignificant compared to Hubble's billion-year observational lookback time despite Voyager traveling farther from Earth than any human-made object?",
        "expected_images": True
    },
    {
        "id": 26,
        "type": "cross_mission_gravity_instrumentalization_comparative",
        "question": "Compare how gravity is instrumentalized across three mission contexts: Voyager's gravity assists (Jupiter's gravity redirects trajectory toward Saturn without fuel), Hubble's gravitational lensing (galaxy cluster gravity magnifies distant galaxies for observation), and Curiosity's precision landing (Mars gravity combined with atmospheric drag enables sky-crane descent). Does this comparison reveal trade-offs between passive observation versus active intervention? Specifically: Why does Voyager's trajectory require irreversible commitment at each gravity assist (Titan flyby precludes further planetary encounters) while Hubble's lensing observations are reversible (can target different clusters), how does the 175-year planetary alignment constraint for Voyager compare to the permanent availability of massive galaxy clusters for Hubble's lensing studies, and why would Curiosity's 20-km landing precision represent active trajectory control (retrorockets, sky-crane) that contrasts with both Voyager's ballistic gravity-assist path and Hubble's passive orbital vantage point—what does this reveal about the relationship between mission objectives (flyby reconnaissance vs. long-term orbital observation vs. surface operations) and the role of gravity in mission design?",
        "expected_images": True
    }
]

In [44]:
retrieval_output=[]
for question in TEST_QUESTIONS:
    retrieval_output.append(retrieval.retrieve(query=question.get("question")))

In [45]:
formatted_output=[]
for output in retrieval_output:
    formatted_output.append(formatter.format(output))

In [46]:
formatted_output[1]

{'query': 'Examine the HH 34 jet observations from 1994, 1998, and 2007 (page 10). What specific visual changes indicate outward motion, and how does the document explain the episodic nature of these stellar outflows?',
 'text_context': '[1] The glowing, clumpy streams of material shown moving left and right in this Hubble image are the signposts of star birth. Collectively named \nHerbig-Haro 47, the speedy outflows have been ejected episodically, like salvos from a cannon, from a young star in the center of the image \nthat is hidden by dust. As they move through space, these outflows create bow shocks and ripples as they collide into other clouds of material \nin the neighborhood of the star.\nThis series of observations by Hubble documents changes in a powerful jet called Herbig-Haro 34 (HH 34) located in the Orion Nebula.\n(Source: highlights_of_hubbles_exploration_of_the_universe.pdf, page 12)\n\n[2] Hubble has also captured energetic jets of glowing gas \nfrom young stars in unp

# LLM

In [47]:
import ollama # Used to load model .
from textwrap import dedent # Used for spacing problems in prompt .
from tabulate import tabulate # Used for creating a table for displaying models .
import subprocess # Used to start ollama server .
import time # For waiting .
import requests # Used to access ollama server .
import base64
import io
from PIL import Image

In [48]:
# Used to initialize a language model and generate responses .
class LocalLLM:
    def __init__(self,model_name:str="gemma3:4b"): # Here is where the model loads .
        self.model_name=model_name
        self.process=self.ollama_server(process="start")

        if not self.is_model_available(self.model_name): # Checking model is available or valid .
            available = [m['model_name'] for m in self.available_models()]
            raise ValueError(
                f"Model '{model_name}' not available. "
                f"Available models: {available}"
            )

# The following function is used to start or stop ollama server .
    def ollama_server(self, process: str):
        if process == "start":
            try:
                requests.get("http://localhost:11434/api/tags", timeout=1) # Checking if ollama server is already started .
                print("Ollama already running")
                return "external"
            except:
                pass

            ollama_process = subprocess.Popen( # Starting ollama server if its not started .
                ["ollama", "serve"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                shell=False
            )

            for _ in range(10): # Checking if sever started .
                try:
                    requests.get("http://localhost:11434/api/tags", timeout=1)
                    print("Ollama server started")
                    return ollama_process
                except:
                    time.sleep(1)

            raise RuntimeError("Ollama failed to start") # If server did not start after many tries than raising error .

        elif process == "stop": # Stopping ollama server .
            if isinstance(self.process, subprocess.Popen): # Checking if server was started using here .
                self.process.terminate()
                self.process.wait()
                print(print("Ollama server successfully stopped ."))
            else:
                print("Ollama was not started by this process") # If python server started externally then notifying it .

            return None

        else:
            raise ValueError("Input can be either 'start' or 'stop'") # Input validation .

# The following function is used check available models in the local device .
    def available_models(self):
        response=ollama.list() # Getting available models .
        models_available=response.models
        models=[]
        for m in models_available: # Getting required information from model .
            models.append({
                "model_name":m.model,
                "parameters":m.details.parameter_size
            })
        if not models:
            return []

        else:
            return models

# The following function is to check if a specific model is available in local device .
    def is_model_available(self, model_name):
        models = self.available_models()
        return any(m['model_name'] == model_name for m in models)

# The following function is to set model for response .
    def set_model(self):
        models=self.available_models()

        if not models: # Checking if models are available to set .
            print("No models available locally .")
            return
        table=[
            [i,m.get("model_name"),m.get("parameters") ]
            for i,m in enumerate(models)
        ]
        print(tabulate( # Displaying available devices .
            table,
            headers=("Serial","Model Name","Parameters"),
            tablefmt="fancy_grid"
        ))

        while True: # Letting users select the model they want .
            try:
                user_input = input("Type model Serial number (or 'q' to cancel): ").strip()

                if user_input.lower() == 'q':
                    print("Cancelled.")
                    return

                option=int(user_input)

                if 0 <= option < len(models): # Input validation .
                    self.model_name = models[option]['model_name']
                    print(f"Model '{self.model_name}' selected.")
                    return
                else:
                    print(f"Invalid serial. Choose 0-{len(models)-1}")
                    print("Choose valid serial number .")
            except ValueError:
                print("Please enter a valid number.")

    # Used to switch model with just the model name .
    def switch_model(self, new_model_name: str):
        if not self.is_model_available(new_model_name):
            available = [m['model_name'] for m in self.available_models()]
            raise ValueError(
                f"Model '{new_model_name}' not available. "
                f"Available models: {available}"
            )

        self.model_name = new_model_name
        print(f"Switched to model: {self.model_name}")
# The following function is used to build prompt using user query and retrieved documents .
    def build_prompt(self, query: str, context: str):
        return f"""
    You are a rigorous scientific analyst.

    Your task is to answer the question using ONLY the provided context and images.
    You must remain strictly evidence-based and avoid speculation.

    You are not allowed to use external knowledge, assumptions, or inferred facts.
    If the available evidence is insufficient, you must clearly explain why.

    ----------------------------------------------------------------
    CORE PRINCIPLES
    ----------------------------------------------------------------

    • Use only the provided text and images.
    • Do not introduce outside knowledge.
    • Do not assume missing details.
    • If the text does not explicitly support the answer, clearly state that the context is insufficient.
    • If an image contradicts the text, clearly explain the inconsistency.
    • If an image is unrelated to the object described in the text, explicitly state that it is not relevant.
    • Before evaluating relevance, verify that the image depicts the SAME object or phenomenon mentioned in the text.

    ----------------------------------------------------------------
    REQUIRED STRUCTURE
    ----------------------------------------------------------------

    1) Textual Evidence Assessment
       - Identify the specific object(s), phenomenon, or event described in the text.
       - Determine whether the text explicitly supports the question.
       - Summarize the exact supporting statements.
       - If the text does not adequately support the answer, explain why and stop.

    2) Image Evaluation
       - Images provided: Yes / No
       - Identify what object or phenomenon is shown in the image.
       - Compare it to the object described in the text.
       - State whether they refer to the same object.
       - If they refer to different objects, clearly state that the image is not relevant.
       - Describe only what is directly visible.
       - Conclude whether the image:
            • Supports the text
            • Contradicts the text
            • Is unrelated or insufficient

    3) Integrated Reasoning
       - Connect the validated textual evidence with any relevant visual evidence.
       - Explain mechanisms, processes, and any numerical details mentioned.
       - Identify logical steps that link evidence to conclusion.
       - Explicitly mention any limitations or missing information.

    4) Final Conclusion
       - Provide a well-structured, natural explanation.
       - Minimum 8–12 detailed sentences.
       - The conclusion must strictly follow from validated evidence.
       - Do not introduce any information not present in the provided material.

    ----------------------------------------------------------------

    Context:
    {context}

    Question:
    {query}

    Answer:
    """.strip()

# The following function is used to convert pillow image type into base64 since llm models can either reads base64 or need image path to access image .
    @staticmethod
    def pil_to_base64(img: Image.Image) -> str:
        buffer = io.BytesIO()
        img = img.convert("RGB")
        img.save(buffer, format="PNG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")


# The following function is used to generate response from language model .
    def generate_response(self,query:str,context:str,images:Optional[List[Image.Image]] = None, stream:bool=True,temperature: float = 0.7,max_tokens: int = 500):
        if not query or not query.strip(): # Query validation .
            raise ValueError("Query cannot be empty")

        if not context or not context.strip():
            if not images:
                raise ValueError("Context cannot be empty when no images are provided")
         # Context validation .

        if len(context) > 10000: # Checking if context is too large .
            print("Warning: Large context may be slow")

        try:
            prompt=self.build_prompt(query,context) # Building a prompt using query and context .

            image_payload = []
            if images:
                for img in images:
                    if isinstance(img.get("image"), Image.Image):
                        image_payload.append(self.pil_to_base64(img.get("image"),))
                    else:
                        raise TypeError("Images must be PIL.Image.Image")
            response=ollama.chat( # Getting response from model .
                model=self.model_name,
                messages = [
                        {"role": "system", "content": "You are a grounded assistant that answers only from provided text and images."},
                        {
                            "role": "user",
                            "content": prompt,
                            **({"images": image_payload} if image_payload else {})
                        }
                    ],
                stream=stream,
                options={
                    'temperature':temperature,
                    "num_predict":max_tokens
                },
                keep_alive=0
            )

            if stream: # Displaying output through streaming .
                print(f"Query: {query}\nAnswer: ", end="")
                full_response = ""
                try:
                    for chunk in response: # Displaying response as model gives output .
                        content = chunk.get("message", {}).get("content", "")
                        if content:
                            print(content, flush=True, end="")
                            full_response+=content
                    print()
                except Exception as e:
                    print(f"\n Error during streaming: {e}") # Exception handling .
                    raise
                return full_response
            else:
                return response["message"]["content"] # If stream is off then giving output all at once .

        except Exception as e:
            raise RuntimeError(f"Error giving response . Error {e}") from e # Exception handling .

In [49]:
llm=LocalLLM() # Initializing model .

Ollama already running


In [50]:
# For timestamp .
import time

In [51]:
# The following function is to get response from llm for given questions .
def llm_response(llm, formatted_output, stream: bool = False):

    response_output = []

    print("=" * 120)
    print(f"Running Model: {llm.model_name}")
    print("=" * 120)

    for idx, output in enumerate(formatted_output):

        query = output.get("query", "")
        text_context = output.get("text_context", "")
        images = output.get("images", [])

        # Append image captions
        if images:
            captions = []
            for i, img in enumerate(images):
                caption = img.get("caption")
                if caption:
                    captions.append(f"[Image {i+1} Caption] {caption}")

            if captions:
                text_context = (
                    text_context
                    + "\n\n[Image Captions]\n"
                    + "\n".join(captions)
                )

        # Logging block
        print("-" * 120)
        print(f"Query #{idx + 1}")
        print(f"Query Text      : {query[:120]}{'...' if len(query) > 120 else ''}")
        print(f"Context Length  : {len(text_context)} characters")
        print(f"Images Retrieved: {len(images)}")

        start_time = time.time()

        response = llm.generate_response(
            query=query,
            context=text_context,
            images=images,
            stream=stream,
            max_tokens=1000,
            temperature=0.4
        )

        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f"Inference Time  : {duration} sec")
        print(f"Response Length : {len(response)} characters")
        print("Status          : Completed")
        print("-" * 120)

        response_output.append({
            "query": query,
            "context": text_context,
            "response": response,
            "images": images,
            "inference_time_sec": duration,
            "response_length": len(response),
            "num_images": len(images)
        })

    print("=" * 120)
    print(f"Completed Model: {llm.model_name}")
    print("=" * 120)

    return response_output

In [52]:
import os
import re
import io
from datetime import datetime
from xml.sax.saxutils import escape

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    Image as RLImage,
    KeepTogether,
    HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib import colors

In [53]:
# The following function is to export results into a PDF .
def export_results_to_pdf(results, model_name: str, output_dir="../data/results"):

    os.makedirs(output_dir, exist_ok=True)

    # -------------------------
    # Timestamp Formatting
    # -------------------------

    now = datetime.now()
    file_timestamp = now.strftime("%Y%m%d_%H%M")
    display_timestamp = now.strftime("%d %B %Y, %I:%M %p")

    safe_model_name = re.sub(r"[^\w\-]", "_", model_name)

    filename = os.path.join(
        output_dir,
        f"{safe_model_name}_rag_results_{file_timestamp}.pdf"
    )

    doc = SimpleDocTemplate(
        filename,
        pagesize=A4,
        rightMargin=45,
        leftMargin=45,
        topMargin=60,
        bottomMargin=50
    )

    elements = []
    styles = getSampleStyleSheet()

    # -------------------------
    # Styles
    # -------------------------

    title_style = ParagraphStyle(
        "TitleStyle",
        parent=styles["Heading1"],
        fontSize=20,
        spaceAfter=20,
        textColor=colors.darkblue
    )

    section_style = ParagraphStyle(
        "SectionStyle",
        parent=styles["Heading3"],
        fontSize=12,
        spaceAfter=6,
    )

    body_style = ParagraphStyle(
        "BodyStyle",
        parent=styles["Normal"],
        fontSize=10.5,
        leading=15,
        spaceAfter=8,
    )

    caption_style = ParagraphStyle(
        "CaptionStyle",
        parent=styles["Normal"],
        fontSize=9,
        textColor=colors.grey,
        leading=12,
        spaceAfter=10,
        alignment=1  # center caption
    )

    # -------------------------
    # Cover Section (No blank page)
    # -------------------------

    elements.append(Paragraph("RAG Evaluation Report", title_style))
    elements.append(Paragraph(f"<b>Model:</b> {safe_model_name}", body_style))
    elements.append(Paragraph(f"<b>Generated:</b> {display_timestamp}", body_style))
    elements.append(Spacer(1, 0.3 * inch))
    elements.append(HRFlowable(width="100%", thickness=1, color=colors.grey))
    elements.append(Spacer(1, 0.5 * inch))

    # -------------------------
    # Main Content
    # -------------------------

    for idx, item in enumerate(results, start=1):

        query = escape(item.get("query", ""))
        context = escape(item.get("context", "")).replace("\n", "<br/>")
        response = escape(item.get("response", "")).replace("\n", "<br/>")

        elements.append(Paragraph(f"Query {idx}", styles["Heading2"]))
        elements.append(Spacer(1, 0.2 * inch))

        elements.append(Paragraph("<b>User Query</b>", section_style))
        elements.append(Paragraph(query, body_style))

        elements.append(Paragraph("<b>Retrieved Context</b>", section_style))
        elements.append(Paragraph(context, body_style))

        elements.append(Paragraph("<b>Model Answer</b>", section_style))
        elements.append(Paragraph(response, body_style))

        # -------------------------
        # Image Section
        # -------------------------

        if item.get("images"):

            image_section = []

            image_section.append(Spacer(1, 0.3 * inch))
            image_section.append(
                Paragraph("<b>Retrieved Image(s)</b>", section_style)
            )
            image_section.append(Spacer(1, 0.2 * inch))

            for img_obj in item["images"]:

                pil_img = img_obj.get("image")
                caption = escape(img_obj.get("caption", ""))

                if pil_img is None:
                    continue

                img_buffer = io.BytesIO()
                pil_img.save(img_buffer, format="PNG")
                img_buffer.seek(0)

                rl_img = RLImage(img_buffer)

                # Controlled Scaling
                max_width = 4.8 * inch
                max_height = 3.5 * inch

                img_width, img_height = pil_img.size

                scale = min(
                    max_width / img_width,
                    max_height / img_height
                )

                rl_img.drawWidth = img_width * scale
                rl_img.drawHeight = img_height * scale
                rl_img.hAlign = "CENTER"

                image_section.append(rl_img)

                if caption:
                    image_section.append(
                        Paragraph(f"<i>{caption}</i>", caption_style)
                    )

                image_section.append(Spacer(1, 0.35 * inch))

            elements.append(KeepTogether(image_section))

        elements.append(PageBreak())

    # -------------------------
    # Footer Page Numbers
    # -------------------------

    def add_page_number(canvas_obj, doc):
        canvas_obj.setFont("Helvetica", 9)
        canvas_obj.drawRightString(
            A4[0] - 40,
            20,
            f"Page {doc.page}"
        )

    doc.build(elements, onLaterPages=add_page_number)

    print(f"Saved PDF to: {filename}")

In [54]:
llm = LocalLLM("gemma3:4b")

models = ["gemma3:4b","ministral-3:3b","llava-phi3:3.8b"]

for model_name in models:
    llm.switch_model(model_name)

    results = llm_response(llm, formatted_output, stream=False)

    export_results_to_pdf(results, model_name=model_name)

Ollama already running
Switched to model: gemma3:4b
Running Model: gemma3:4b
------------------------------------------------------------------------------------------------------------------------
Query #1
Query Text      : Analyze the three images of Jupiter's Great Red Spot from 1995, 2009, and 2014 (page 7). What visual evidence demonstrat...
Context Length  : 3015 characters
Images Retrieved: 1
Inference Time  : 21.35 sec
Response Length : 3179 characters
Status          : Completed
------------------------------------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------------------------------------
Query #2
Query Text      : Examine the HH 34 jet observations from 1994, 1998, and 2007 (page 10). What specific visual changes indicate outward mo...
Context Length  : 2272 characters
Images Retrieved: 1
Inference Time  : 18.14 sec
Response Length : 2587 character